1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

2. Load the Dataset

In [ ]:
# Load your data
df = pd.read_csv('/content/tweets.csv')
# Display the first few rows to ensure it loaded correctly
print(df.head())

   id  label                                              tweet
0   1      0  #fingerprint #Pregnancy Test https://goo.gl/h1...
1   2      0  Finally a transparant silicon case ^^ Thanks t...
2   3      0  We love this! Would you go? #talk #makememorie...
3   4      0  I'm wired I know I'm George I was made that wa...
4   5      1  What amazing service! Apple won't even talk to...


In [ ]:
df.isnull().sum()

,0
id,0
label,0
tweet,0


In [ ]:

duplicates = df[df.duplicated()]

print(f"Number of duplicate rows: {len(duplicates)}")
print(duplicates.head())

Number of duplicate rows: 0
Empty DataFrame
Columns: [id, label, tweet]
Index: []


In [ ]:
df.duplicated().sum()

np.int64(0)

3. Data Preprocessing

In [ ]:
import re

def clean_text(text):
    text = re.sub(r'http\S+', '', text) # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove special characters/numbers
    text = text.lower().strip() # Lowercase
    return text

df['cleaned_tweet'] = df['tweet'].apply(clean_text)

In [ ]:
#4. Split Data into Train and Test Sets

In [ ]:
X = df['cleaned_tweet']
y = df['label'] # Assuming 'label' is 1 for positive, 0 for negative

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

5. Feature Extraction (Vectorization)

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

6. Model Building and Training

In [ ]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

LogisticRegression()

7. Model Evaluation (Accuracy)

In [ ]:
y_pred = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Model Accuracy: 87.82%

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.94      0.92      1152
           1       0.82      0.72      0.76       432

    accuracy                           0.88      1584
   macro avg       0.86      0.83      0.84      1584
weighted avg       0.88      0.88      0.88      1584



8.Prediction with new data

In [ ]:
def predict_sentiment(new_tweet):
    # 1. Clean the text using the function we defined earlier
    cleaned_text = clean_text(new_tweet)

    # 2. Transform the text using the same TF-IDF vectorizer used in training
    tweet_vector = tfidf.transform([cleaned_text])

    # 3. Use the model to predict
    prediction = model.predict(tweet_vector)

    # 4. Convert the numerical label back to a readable string
    if prediction[0] == 0:
        return "Positive Sentiment"
    else:
        return "Negative Sentiment"

# --- Test it out ---
test_tweet = "What amazing service! Mi won't even talk to me about a question I have unless I pay them $19.95 for their stupid support!"
result = predict_sentiment(test_tweet)

print(f"Tweet: {test_tweet}")
print(f"Result: {result}")

Tweet: What amazing service! Mi won't even talk to me about a question I have unless I pay them $19.95 for their stupid support!
Result: Negative Sentiment
